In [ ]:
from datetime import datetime
import base64
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pytz

In [ ]:
# Load the dataset
apps_df = pd.read_csv('Play Store Data.csv')
reviews_df = pd.read_csv('User Reviews.csv')
apps_df.head(3)

In [ ]:
# Helper function to parse size values to MB
def parse_size_to_mb(val):
  val = str(val).strip()
  if val.endswith('M'):
    try:
      return float(val[:-1])
    except ValueError:
      return None
  elif val.endswith('k') or val.endswith('K'):
    try:
      return float(val[:-1]) / 1024.0
    except ValueError:
      return None
  return None

In [ ]:
# Drop missing values
apps_df = apps_df.dropna(subset=['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Last Updated']).copy()

# Clean 'Installs'
apps_df['Installs'] = apps_df['Installs'].str.replace(',', '').str.replace('+', '')
apps_df = apps_df[apps_df['Installs'].str.isnumeric() == True]
apps_df['Installs'] = apps_df['Installs'].astype(int)

# Apply the clean parsing function
apps_df['Size_MB'] = apps_df['Size'].apply(parse_size_to_mb)

# Clean 'Rating' and 'Reviews'
apps_df['Rating'] = pd.to_numeric(apps_df['Rating'], errors='coerce')
# apps_df['Rating'] = apps_df['Rating'].apply(lambda x: x if 0 <= x <= 5 else np.nan)
apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'], errors='coerce')

# Convert 'Last Updated' to dates
apps_df['Last Updated'] = pd.to_datetime(apps_df['Last Updated'], errors='coerce')

print(f"Rows after cleaning: {len(apps_df)}")
# Check the first few rows
apps_df.head(3)

In [ ]:
# Filter apps: Size >= 10 MB AND Last Updated in January
filtered_df = apps_df[(apps_df['Size_MB'] >= 10.0) & (apps_df['Last Updated'].dt.month == 1)]

# Check how many apps matched our filter
print(f'Total apps matching criteria: {len(filtered_df)}')
filtered_df[['App', 'Category', 'Size_MB', 'Last Updated']].head(10)

In [ ]:
# Group by Category and calculate metrics
category_df = (
    filtered_df.groupby('Category').agg(
        Total_Installs=('Installs', 'sum'),
        Avg_Rating=('Rating', 'mean'),
        Total_Reviews=('Reviews', 'sum'),
    ).reset_index()
)

# Filter categories with Average Rating >= 4.0 and sort by Total Installs to get the Top 10
top10_df = category_df[category_df['Avg_Rating'] >= 4.0].sort_values(by='Total_Installs', ascending=False).head(10)

# Display the top 10 categories table
top10_df

In [ ]:
# Time check (3:00 PM to 5:00 PM IST)
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()

start_time = datetime.strptime('15:00:00', '%H:%M:%S').time()
end_time = datetime.strptime('17:00:00', '%H:%M:%S').time()

is_within_time = start_time <= current_time < end_time

# Prepare HTML elements
total_apps_matched = len(filtered_df)
cat_rows_html = ''
for idx, row in top10_df.iterrows():
  cat_rows_html += (
      f"<tr><td>{row['Category']}</td><td>{row['Avg_Rating']:.2f}</td><td>{int(row['Total_Reviews']):,}</td><td>{int(row['Total_Installs']):,}</td></tr>\n"
  )

# Defined file paths
graph_path = 'task_1_graph.html'
dashboard_path = 'task_1_dashboard.html'

In [ ]:
# Generating the graph using Plotly and saving it as task_1_graph.html silently if within the allowed time window
if is_within_time:
  fig = plt.figure(figsize=(12, 6))
  ax1 = fig.add_subplot(111)
  ax2 = ax1.twinx()  # Create secondary y-axis

  x = np.arange(len(top10_df['Category']))  # Category positions
  width = 0.35  # Equal width for all bars

  # Plot Avg Rating on Primary (Left) Y-axis
  rects1 = ax1.bar(
      x - width / 2,
      top10_df['Avg_Rating'],
      width=width,
      color='royalblue',
      label='Avg Rating',
  )

  # Plot Total Reviews on Secondary (Right) Y-axis
  rects2 = ax2.bar(
      x + width / 2,
      top10_df['Total_Reviews'],
      width=width,
      color='darkorange',
      label='Total Reviews',
  )

  # Fix X-axis labels and tick positions directly under bar pairs
  ax1.set_xticks(x)
  ax1.set_xticklabels(top10_df['Category'], rotation=45, ha='right')
  ax1.set_xlim(-0.6, len(top10_df) - 0.4)

  # Axis Labels and Limits
  ax1.set_ylabel('Average Rating (1-5)')
  ax1.set_ylim(0, 5)
  ax2.set_ylabel('Total Reviews (in Millions)')

  # Combined Legend
  lines1, labels1 = ax1.get_legend_handles_labels()
  lines2, labels2 = ax2.get_legend_handles_labels()
  ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

  plt.title(
      'Top 10 App Categories by Installs: Average Rating vs Total Reviews'
      ' (January Updates, Size ≥ 10 MB)'
  )
  plt.tight_layout()

  # Save Matplotlib figure into task_1_graph.html as Base64 HTML
  img_buffer = io.BytesIO()
  plt.savefig(img_buffer, format='png', bbox_inches='tight')
  img_buffer.seek(0)
  img_base64 = base64.b64encode(img_buffer.getvalue()).decode('utf-8')
  plt.close()

  graph_html_content = f"""<!DOCTYPE html>
<html>
<head>
    <title>Task 1 Graph</title>
</head>
<body style="text-align: center; background-color: #f4f6f9; padding: 20px;">
    <h2>Top 10 App Categories Visualization</h2>
    <img src="data:image/png;base64,{img_base64}" alt="Matplotlib Graph" style="max-width: 100%; height: auto; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1);">
</body>
</html>"""

  with open(graph_path, 'w', encoding='utf-8') as f:
    f.write(graph_html_content)

  status_section_html = f"""
    <div class="card">
        <h2>Graph Status</h2>
        <p style="color: green; font-weight: bold;">Status: Graph Successfully Generated</p>
        <p>The Matplotlib graph has been generated and saved to <b>{graph_path}</b> because the execution occurred within the allowed window (3:00 PM - 5:00 PM IST).</p>
        <p><a href="{graph_path}" target="_blank" style="display: inline-block; padding: 10px 15px; background-color: #1a73e8; color: white; text-decoration: none; border-radius: 5px;">Open Graph</a></p>
    </div>
  """

else:
  status_section_html = f"""
    <div class="card">
        <h2>Graph Status</h2>
        <p style="color: red; font-weight: bold;">Status: Graph Restricted</p>
        <p>The graph is only generated and accessible between <b>3:00 PM IST and 5:00 PM IST</b>. The graph file ({graph_path}) was not created. Please rerun the code during the allowed time window.</p>
    </div>
  """

In [ ]:
# Generating the dashboard HTML content
dashboard_html_content = f"""<!DOCTYPE html>
<html>
<head>
    <title>Play Store Installs Dashboard - Task 1</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 30px;
            background-color: #f4f6f9;
            color: #333;
        }}
        h1 {{ color: #1a73e8; }}
        .card {{
            background: white;
            padding: 20px;
            margin-bottom: 20px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 10px;
        }}
        th, td {{
            padding: 10px;
            border: 1px solid #ddd;
            text-align: left;
        }}
        th {{ background-color: #f0f4f8; }}
    </style>
</head>
<body>

    <h1>Play Store App Analysis Dashboard (Task 1)</h1>

    <div class="card">
        <h2>Dashboard Summary</h2>
        <p><b>Filter Rules Applied:</b> App Size &ge; 10 MB, Updated in January, and Category Average Rating &ge; 4.0.</p>
    </div>

    <div class="card">
        <h2>Top 10 Categories</h2>
        <table>
            <tr>
                <th>Category Name</th>
                <th>Avg Rating</th>
                <th>Total Reviews</th>
                <th>Total Installs</th>
            </tr>
            {cat_rows_html}
        </table>
    </div>

    {status_section_html}

</body>
</html>
"""

with open(dashboard_path, 'w', encoding='utf-8') as f:
  f.write(dashboard_html_content)